# Chapter 4 — Factory Patterns
## *Baking with OO Goodness*

Three related patterns covered in this chapter:
1. **Simple Factory** — not a formal pattern, but a useful idiom.
2. **Factory Method** — defines an interface for creating an object, but lets subclasses decide which class to instantiate.
3. **Abstract Factory** — provides an interface for creating **families** of related objects.

### OO Design Principle
- **Dependency Inversion Principle:** Depend on abstractions, not concrete classes.  
  High-level modules should not depend on low-level modules; both should depend on abstractions.

### Book context
A pizza store franchise: `PizzaStore` subclasses (`NYPizzaStore`, `ChicagoPizzaStore`) each produce regionally specific pizzas, but share the same `order_pizza()` workflow.

---
## Part 1 — Simple Factory

In [ ]:
# Simple Factory: centralize object creation, but no subclassing

class Pizza:
    def __init__(self, name: str): self.name = name
    def prepare(self): print(f"Preparing {self.name}")
    def bake(self):    print(f"Baking {self.name}")
    def cut(self):     print(f"Cutting {self.name}")
    def box(self):     print(f"Boxing {self.name}")

class CheesePizza(Pizza):
    def __init__(self): super().__init__("Cheese Pizza")

class VeggiePizza(Pizza):
    def __init__(self): super().__init__("Veggie Pizza")

class PepperoniPizza(Pizza):
    def __init__(self): super().__init__("Pepperoni Pizza")


class SimplePizzaFactory:
    _registry: dict[str, type] = {
        "cheese":    CheesePizza,
        "veggie":    VeggiePizza,
        "pepperoni": PepperoniPizza,
    }

    @staticmethod
    def create_pizza(kind: str) -> Pizza:
        cls = SimplePizzaFactory._registry.get(kind)
        if cls is None:
            raise ValueError(f"Unknown pizza type: {kind}")
        return cls()


pizza = SimplePizzaFactory.create_pizza("cheese")
pizza.prepare()
pizza.bake()

---
## Part 2 — Factory Method Pattern

In [ ]:
from abc import ABC, abstractmethod

# ── Product hierarchy ────────────────────────────────────────────────────────
class Pizza(ABC):
    name: str = ""

    def prepare(self): print(f"  Preparing {self.name}")
    def bake(self):    print(f"  Baking {self.name} at 350°F for 25 min")
    def cut(self):     print(f"  Cutting {self.name} diagonally")
    def box(self):     print(f"  Placing {self.name} in box")

class NYStyleCheesePizza(Pizza):
    name = "NY Style Sauce and Cheese Pizza"

class NYStyleVeggiePizza(Pizza):
    name = "NY Style Veggie Pizza"

class ChicagoStyleCheesePizza(Pizza):
    name = "Chicago Style Deep Dish Cheese Pizza"
    def cut(self): print(f"  Cutting {self.name} into squares")

class ChicagoStyleVeggiePizza(Pizza):
    name = "Chicago Style Veggie Pizza"
    def cut(self): print(f"  Cutting {self.name} into squares")


# ── Creator hierarchy ────────────────────────────────────────────────────────
class PizzaStore(ABC):
    # Template Method + Factory Method combo
    def order_pizza(self, kind: str) -> Pizza:
        pizza = self.create_pizza(kind)   # factory method
        pizza.prepare()
        pizza.bake()
        pizza.cut()
        pizza.box()
        return pizza

    @abstractmethod
    def create_pizza(self, kind: str) -> Pizza: ...


class NYPizzaStore(PizzaStore):
    def create_pizza(self, kind):
        return {"cheese": NYStyleCheesePizza,
                "veggie": NYStyleVeggiePizza}[kind]()

class ChicagoPizzaStore(PizzaStore):
    def create_pizza(self, kind):
        return {"cheese": ChicagoStyleCheesePizza,
                "veggie": ChicagoStyleVeggiePizza}[kind]()


print("=== NY Store ===")
ny = NYPizzaStore()
p = ny.order_pizza("cheese")
print(f"\nEthan ordered: {p.name}")

print("\n=== Chicago Store ===")
chi = ChicagoPizzaStore()
p2 = chi.order_pizza("cheese")
print(f"\nJoel ordered: {p2.name}")

---
## Part 3 — Abstract Factory Pattern

In [ ]:
# Abstract Factory: families of related ingredients

class PizzaIngredientFactory(ABC):
    @abstractmethod
    def create_dough(self) -> str: ...
    @abstractmethod
    def create_sauce(self) -> str: ...
    @abstractmethod
    def create_cheese(self) -> str: ...


class NYIngredientFactory(PizzaIngredientFactory):
    def create_dough(self):  return "Thin Crust Dough"
    def create_sauce(self):  return "Marinara Sauce"
    def create_cheese(self): return "Reggiano"

class ChicagoIngredientFactory(PizzaIngredientFactory):
    def create_dough(self):  return "Thick Crust Dough"
    def create_sauce(self):  return "Plum Tomato Sauce"
    def create_cheese(self): return "Mozzarella"


class CheesePizzaV2:
    def __init__(self, factory: PizzaIngredientFactory):
        self.dough  = factory.create_dough()
        self.sauce  = factory.create_sauce()
        self.cheese = factory.create_cheese()
        self.name   = "Cheese Pizza"

    def __repr__(self):
        return (f"{self.name}: {self.dough}, {self.sauce}, {self.cheese}")


ny_pizza  = CheesePizzaV2(NYIngredientFactory())
chi_pizza = CheesePizzaV2(ChicagoIngredientFactory())
print(ny_pizza)
print(chi_pizza)

## Interview cheat-sheet

| | Simple Factory | Factory Method | Abstract Factory |
|---|---|---|---|
| **Mechanism** | One class, one method | Subclass overrides creator | Interface for a family of products |
| **Extends via** | — | Inheritance | Composition |
| **Creates** | One product type | One product (per subclass) | Multiple related products |
| **Use when** | Centralize creation | Subclasses decide the type | System must be independent of product families |

**Pattern family:** Creational